# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import BatchGemini, Batch
from pricer.items import Item

load_dotenv(override=True)

True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [4]:
LITE_MODE = True

In [5]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [7]:
dataset['train'][0]

TypeError: string indices must be integers, not 'str'

In [6]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [8]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [9]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [16]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]

response = completion(
    model="openrouter/openai/gpt-oss-20b",
    messages=messages,
    extra_body={"reasoning": {"effort": "low"}}
)

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Schlage F59 Half Door Knob, Oil Rubbed Bronze  
Category: Hardware  
Brand: Schlage  
Description: A precision‑engineered interior half‑door knob with integrated deadbolt offering enhanced security and peace of mind.  
Details: Features an oil‑rubbed bronze finish, easy‑install design, and a lifetime mechanical and finish warranty.

Input tokens: 447
Output tokens: 94
Cost: 0.004 cents


In [9]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="openai/gpt-5-nano", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")

Title: Schlage Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half)
Category: Home Improvement
Brand: Schlage
Description: Interior half of a two-piece Andover handle set with deadbolt in oil rubbed bronze for secure, stylish entryway doors.
Details: Easy-to-install interior knob and deadbolt half in oil rubbed bronze; non-handed knob; requires matching F58/handle set components to complete; includes standard 4" prep.

Input tokens: 382
Output tokens: 297
Cost: 0.014 cents


In [25]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full}
]

response = completion(
    model="gemini/gemini-flash-lite-latest",
    messages=messages,
    thinking_effort="low",
)

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")



Provider List: https://docs.litellm.ai/docs/providers

Title: Andover Interior Door Knob and Deadbolt Half Set
Category: Hardware
Brand: Schlage
Description: This is the interior half of an Andover style knob and deadbolt set in an Oil Rubbed Bronze finish.
Details: Features a non-handed knob style and includes a lifetime mechanical and finish warranty.

Input tokens: 381
Output tokens: 64
Cost: 0.006 cents

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



In [ ]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
MODEL = "openai/gpt-oss-20b"


In [ ]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [ ]:
items[0]

In [ ]:
make_jsonl(items[0])

In [ ]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:

with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
result = groq.batches.retrieve(response.id)
result

In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [9]:
Batch.create(items, LITE_MODE)

Created 22 batches


In [12]:
for b in Batch.batches:
    print(f"{b.file_id}, {b.output_file_id}, {b.batch_id}")

file-Ru4XW4k9vdcHand2VaYDrK, None, batch_696eaa932cc8819089ca84ee2100c6dc
file-SFteoWpEJSLFtqvQCZmhbp, None, batch_696eaa98937481908300493d4ecb56d2
file-VueW8BV21NzG7ho5iUFo5o, None, batch_696eaa9c8a688190b36c4aa520c2978c
file-GdpCjhb2aWXArrnouf1mch, None, batch_696eaaa090508190b592a55449e60df4
file-3kwAVqsudnLb88hncvWdyn, None, batch_696eaaa48b8c8190a880f7e2b399c2f6
file-RwkMahkYXbnSsAqVYXeAzW, None, batch_696eaaa8875481908c4eb89a629d3022
file-RjfszFZkZnhG47PPb9n18P, None, batch_696eaaaca0ac8190aa2ed12d137625ae
file-7kehYLekatvgALDdtRfbcU, None, batch_696eaab0bae481908d5bb84a07f213b3
file-MY5s3L14rY2hJF5Hc8qndd, None, batch_696eaab506048190bef8e3761b130184
file-KvuXi3JoGsHZWTDftSHrFB, None, batch_696eaab902b48190b8118aacd8b2ca00
file-JmqhRvVi5JavjvJHy3dQPZ, None, batch_696eaabd70d4819093112f6070e0f719
file-GrNbdX67qADUZfeSKmR1QC, None, batch_696eaac2a3f08190a1a1304dd2648306
file-27kQP9Q1AvZpyXCBmpKSqP, None, batch_696eaac6d5b88190a3f943c3feb3c8e2
file-Qi2guQ9Mq1RpoT8W7HKHJs, None, bat

In [ ]:
# fetching manually from ID file

In [11]:
Batch.run()

  0%|          | 0/22 [00:00<?, ?it/s]

Submitted 22 batches


In [14]:
Batch.fetch()

  0%|          | 0/22 [00:00<?, ?it/s]

Finished 0 of 22 batches


In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
